In [ ]:
from google.colab import drive
drive.mount('/content/drive')


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
# Copying the segmentation folder from the slow Drive to the fast Colab memory
!cp -r /content/drive/MyDrive/Brain_Tumor_Project/brisc2025/segmentation_task /content/fast_seg_data

import os
print("Checking fast memory...")
print("Train folders:", os.listdir('/content/fast_seg_data/train'))

print("\nCopying complete! The pictures and masks are ready.")

cp: cannot access '/content/drive/MyDrive/Brain_Tumor_Project/brisc2025/segmentation_task/train/images': Software caused connection abort
cp: cannot stat '/content/drive/MyDrive/Brain_Tumor_Project/brisc2025/segmentation_task/train/masks': No such file or directory
cp: cannot stat '/content/drive/MyDrive/Brain_Tumor_Project/brisc2025/segmentation_task/test': No such file or directory
Checking fast memory...
Train folders: ['images']

Copying complete! The pictures and masks are ready.


In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from PIL import Image
import os

# 1. Create a custom rulebook for pairing images and masks
class BrainTumorDataset(Dataset):
    def __init__(self, images_dir, masks_dir):
        self.images_dir = images_dir
        self.masks_dir = masks_dir
        # Sort them alphabetically so image 01 matches mask 01 exactly
        self.images = sorted(os.listdir(images_dir))
        self.masks = sorted(os.listdir(masks_dir))

        # Rule for resizing both pictures
        self.transform = transforms.Compose([
            transforms.Resize((224, 224)),
            transforms.ToTensor()
        ])

    def __len__(self):
        return len(self.images)

    def __getitem__(self, idx):
        # Grab the matching pair
        img_path = os.path.join(self.images_dir, self.images[idx])
        mask_path = os.path.join(self.masks_dir, self.masks[idx])

        # Open the pictures
        image = Image.open(img_path).convert("RGB") # Color format for U-Net
        mask = Image.open(mask_path).convert("L")   # Black and white for the mask

        # Resize them and turn them into math
        image = self.transform(image)
        mask = self.transform(mask)

        return image, mask

# 2. Point to our fast memory folders
train_img_dir = '/content/fast_seg_data/train/images'
train_mask_dir = '/content/fast_seg_data/train/masks'

# 3. Apply the rulebook to create our dataset
train_dataset = BrainTumorDataset(train_img_dir, train_mask_dir)

# 4. Make the flashcard stacks (we use 16 pictures at a time here to save memory)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)

print(f"Pairing complete! The AI has {len(train_dataset)} MRI-and-Mask pairs to study.")

FileNotFoundError: [Errno 2] No such file or directory: '/content/fast_seg_data/train/masks'

In [ ]:
# 1. Install the special segmentation library
!pip install segmentation-models-pytorch -q

import segmentation_models_pytorch as smp
import torch

print("\nDownloading the U-Net brain...")

# 2. Build the U-Net!
# We are actually giving it a mini-ResNet (resnet34) as a head start for the first half
unet_model = smp.Unet(
    encoder_name="resnet34",        # The smart student helping it learn
    encoder_weights="imagenet",     # Pre-trained on basic shapes
    in_channels=3,                  # Looking at RGB (3-color) pictures
    classes=1                       # Outputting exactly 1 thing: the tumor mask
)

# 3. Put it on the GPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
unet_model = unet_model.to(device)

print("U-Net Brain is ready and running on:", device)

In [ ]:
import torch.nn as nn
import torch.optim as optim

# 1. The Grader (Binary Cross-Entropy for True/False pixel guessing)
criterion = nn.BCEWithLogitsLoss()

# 2. The Tutor (Adam optimizer, exactly like last time)
optimizer = optim.Adam(unet_model.parameters(), lr=0.001)

print("Grader and Tutor for U-Net are ready!")

In [ ]:
# How many times to study the whole stack
epochs = 3

print("Starting U-Net training! This will take a few minutes...\n")

for epoch in range(epochs):
    print(f"--- Starting Epoch {epoch+1} of {epochs} ---")

    # 1. Turn on "learning mode"
    unet_model.train()
    total_mistakes = 0.0

    # 2. Look at the pairs of pictures 16 at a time
    for i, (images, masks) in enumerate(train_loader):

        # Send pictures and masks to the fast GPU engine
        images = images.to(device)
        masks = masks.to(device)

        # Tell the tutor to erase notes from the last batch
        optimizer.zero_grad()

        # The AI looks at the MRI and tries to draw the tumor mask!
        guesses = unet_model(images)

        # The Grader compares the AI's drawing pixel-by-pixel to the real mask
        mistake = criterion(guesses, masks)

        # The Tutor figures out how to fix the brain
        mistake.backward()

        # The Tutor actually turns the knobs to make the drawing better
        optimizer.step()

        total_mistakes += mistake.item()

        # Print an update every 50 batches so we know it's working
        if i % 50 == 0:
            print(f"Batch {i}: U-Net is practicing drawing...")

    # 3. Average score for the epoch
    average_mistake = total_mistakes / len(train_loader)
    print(f"Epoch {epoch+1} finished! Average Mistake Score: {average_mistake:.4f}\n")

print("Training complete! The U-Net can now draw tumor boundaries.")

In [ ]:
import torch

# Save the segmentation brain right next to your classification brain
save_path = '/content/drive/MyDrive/Brain_Tumor_Project/segmenter.pth'
torch.save(unet_model.state_dict(), save_path)

print("U-Net Brain safely saved to Google Drive!")

In [ ]:
!pip install streamlit -q
!npm install localtunnel

In [ ]:
%%writefile app.py
import streamlit as st
import torch
import torchvision.transforms as transforms
from PIL import Image
import torchvision.models as models
import segmentation_models_pytorch as smp
import numpy as np
from pytorch_grad_cam import GradCAM
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

# --- 1. SET UP THE PAGE DESIGN ---
st.set_page_config(page_title="AI Brain Tumor Diagnostic Tool", layout="wide")
st.title("🧠 AI-Powered Brain Tumor Diagnostic Dashboard")
st.markdown("Upload a T1-weighted MRI scan to instantly receive tumor classification and precise spatial segmentation.")

# --- 2. LOAD THE AI BRAINS ---
@st.cache_resource
def load_models():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    classifier = models.resnet50(weights=None)
    num_ftrs = classifier.fc.in_features
    classifier.fc = torch.nn.Linear(num_ftrs, 4)
    classifier.load_state_dict(torch.load('/content/drive/MyDrive/Brain_Tumor_Project/classifier.pth', map_location=device))
    classifier = classifier.to(device)
    classifier.eval()

    segmenter = smp.Unet(encoder_name="resnet34", encoder_weights=None, in_channels=3, classes=1)
    segmenter.load_state_dict(torch.load('/content/drive/MyDrive/Brain_Tumor_Project/segmenter.pth', map_location=device))
    segmenter = segmenter.to(device)
    segmenter.eval()

    return classifier, segmenter, device

classifier, segmenter, device = load_models()
classes = ['Glioma', 'Meningioma', 'No Tumor', 'Pituitary']

# --- 3. UPLOAD AND PROCESS ---
st.sidebar.header("Patient Data Input")
uploaded_file = st.sidebar.file_uploader("Upload MRI Image (JPG/PNG)", type=["jpg", "png", "jpeg"])

if uploaded_file is not None:
    image = Image.open(uploaded_file).convert('RGB')

    transform = transforms.Compose([
        transforms.Resize((224, 224)),
        transforms.ToTensor(),
    ])
    input_tensor = transform(image).unsqueeze(0).to(device)

    # --- 4. PREDICTIONS & GRAD-CAM ---
    with torch.no_grad():
        class_out = classifier(input_tensor)
        _, predicted_class = torch.max(class_out, 1)
        diagnosis = classes[predicted_class.item()]

        mask_out = segmenter(input_tensor)
        mask_pred = torch.sigmoid(mask_out).squeeze().cpu().numpy()
        mask_binary = (mask_pred > 0.5).astype(np.uint8)

    # Generate Grad-CAM Heatmap
    target_layers = [classifier.layer4[-1]]
    cam = GradCAM(model=classifier, target_layers=target_layers)
    targets = [ClassifierOutputTarget(predicted_class.item())]
    grayscale_cam = cam(input_tensor=input_tensor, targets=targets)[0, :]

    img_float = np.array(image.resize((224, 224))) / 255.0
    cam_image = show_cam_on_image(img_float, grayscale_cam, use_rgb=True)

    # --- 5. DASHBOARD LAYOUT ---
    st.markdown("---")
    col1, col2, col3 = st.columns(3)

    with col1:
        st.subheader("1. Original MRI")
        st.image(image, use_container_width=True)
        st.info(f"**Classification Result:** {diagnosis}")

    with col2:
        st.subheader("2. Tumor Localization (Mask)")
        st.image(mask_binary * 255, use_container_width=True, clamp=True)
        if diagnosis == 'No Tumor':
            st.success("No significant tumor regions detected.")
        else:
            st.warning("Tumor boundaries identified.")

    with col3:
        st.subheader("3. Explainability (Grad-CAM)")
        st.image(cam_image, use_container_width=True)
        st.caption("*Grad-CAM feature map highlighting the regions influencing the classification decision.*")

else:
    st.info("Please upload an MRI scan from the sidebar to begin the analysis.")

Writing app.py


In [ ]:
!pip install grad-cam -q

In [ ]:
# Get the IP address again (just in case)
!wget -q -O - ipv4.icanhazip.com

# Start Streamlit with security blockers turned off
!streamlit run app.py --server.enableCORS false --server.enableXsrfProtection false & npx localtunnel --port 8501

136.109.191.132
⠙⠹⠸⠼

⠴⠦⠧⠇your url is: https://itchy-cycles-dress.loca.lt
2026-09-24 09:35:01.450 Uvicorn server started on :::8502

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8502
  Network URL: http://172.28.0.12:8502
  External URL: http://136.109.191.132:8502

